# 健常DDR：PA/ML YOLO Pose 再検証 ＋ #005 Matsuda LCEA自動計測
**2026-10-23 | 北村先生依頼・学会発表用 | Google Colab GPU**

### 目的
1. **同じ健常COCO ZIPからPA 10点／ML 9点**を個別に新規学習し、**患者（動画）単位の交差検証**を行う。
2. 未知患者の **Pose precision, recall, mAP50, mAP50-95** と **Box系指標**を別々に集計する。
3. keypointの位置誤差（ユークリッド距離 px）と基準線角度誤差（°）を **MAE・RMSE** の両方で報告。**患者別・landmark別・fold別**も保存する。
4. **#005 Matsuda PA/MLは学習から除外したfoldのAI**で手動 annotation vs AI keypoint Figure（PNG/SVG）を作る。
5. **#005 Matsuda PAを除外して学習した専用左右ROIモデル**とPA全骨盤AIでLCEA自動計測し、角度数値入りMP4・CSVを作る。推論にGTは使わない。

### 重要なデータ監査結果（今回添付ZIP）
- **PA：健常9名／174フレーム**（001–006,008–010）。**011 PAはこのZIPにない**。
- **ML：健常10名／192フレーム**（001–006,008–011）。
- 旧5健常PAは新ZIP内の001–005と同一画像。**過去の `Annotation PA.zip` は追加投入しない**（重複防止）。
- `ML 003` では `PT R` の重複annotation、`PA 009` では `Medial sourci` の欠損フレームがある。QC表に記録し、見えない点は `v=0` で扱う。
- 出力値は新しい学習が完走するまでは存在しません。数値を発表する前に患者数・fold数・欠測率と併せてQC。

**評価上の注意:** Pose mAPは物体検出型のOKS指標（Ultralytics実装）で、点誤差のpx値とは別物です。連続フレームは独立患者として扱いません。PA/MLの計測対象も別です。


## 0. GPUと依存ライブラリ

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "opencv-python-headless>=4.8" "pandas>=2.2" numpy matplotlib pyyaml pillow scikit-learn tqdm

In [ ]:
import os, re, io, json, zipfile, shutil, hashlib, math, gc, warnings, random, subprocess
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, yaml, torch
from tqdm.auto import tqdm
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from ultralytics import YOLO
from IPython.display import display, Image, Video
random.seed(42); np.random.seed(42)
print('Ultralytics / GPU:',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. 設定・ZIPアップロード
**入力は `AI-Hip Gate Analysis.zip` のみ。** 以前の股関節重みとDDH ZIPは使用しません。Google Drive利用時は `SOURCE_ZIP` と `ROOT` を適宜変更。長時間計算に備えてDrive下の作業ディレクトリを推奨します。

`CV_MODE='group5'` は5-fold患者単位GroupKFold（実用設定）、`'lovo'` は各患者を1動画ずつ除外（PA 9fold＋ML 10fold）。**以前の5例LOVOと同等の分割単位を重視するなら `'lovo'`** に変更します。両解析の平均mAPの直接比較は、症例数・fold設計・学習条件が異なるため控えてください。


In [ ]:
SOURCE_ZIP = Path('/content/AI-Hip Gate Analysis.zip')
ROOT = Path('/content/DDR_healthy10_pose_20261023')
# Drive例: ROOT=Path('/content/drive/MyDrive/DDR_healthy10_pose_20261023')
CV_MODE = 'group5'  # 'group5' or 'lovo'
RUN_TRAIN = True
RUN_LOCAL_005 = True
FORCE_RETRAIN = False
MODEL_INITIALIZATION = 'yolo11n-pose.pt'  # 一般COCO事前重み。過去の股関節重みは不使用
# 完全ランダム初期化の実験: 'yolo11n-pose.yaml' とする（比較結果には初期値を明記）。
EPOCHS = 110; PATIENCE = 22; IMGSZ = 960; BATCH = 4
LOCAL_EPOCHS = 125; LOCAL_IMGSZ = 640; LOCAL_BATCH = 8
PRED_CONF = 0.10; MIN_KP_CONF = 0.20
VAL_PATIENT_OVERRIDE = {}  # 必要時のみ指定。例: {'PA': '010'}
DISPLAY_FPS = 5  # 画像列から再構成するデモ動画の表示用FPS。DDRの元fpsではない。
ORIGINAL_005_PA_MP4 = None  # 元DDR動画のパスがある場合: '/content/005_Matsuda_PA.mp4'
OUTPUT_ZIP = Path('/content/DDR_healthy10_PA_ML_GroupCV_Matsuda005_results.zip')
if not SOURCE_ZIP.exists():
    raise FileNotFoundError(f'{SOURCE_ZIP} をColabにアップロードし、SOURCE_ZIPを確認してください。')
ROOT.mkdir(parents=True,exist_ok=True)
DATA=ROOT/'source'; DATA.mkdir(exist_ok=True)
QC=ROOT/'QC'; QC.mkdir(exist_ok=True)
FIG=ROOT/'figures'; FIG.mkdir(exist_ok=True)
VIDEOS=ROOT/'videos';VIDEOS.mkdir(exist_ok=True)
RUNS=ROOT/'runs';RUNS.mkdir(exist_ok=True)


## 2. COCO → 共通landmark辞書／YOLO形式変換
COCOのcategory ID・keypoint indexは、**各ZIPの `categories[].keypoints` 名を参照**して解決します。`LCEA L/R`, `physeal scar` は今回の従来PA 10点評価に混ぜません。duplicate annotationはログに残して可視点の多い方を採用します。

In [ ]:
PA_KPS = ['lateral_sourcil_R','lateral_sourcil_L','medial_sourcil_R','medial_sourcil_L',
          'teardrop_R','teardrop_L','femoral_head_R','femoral_head_L','sacrum_R','sacrum_L']
ML_KPS = ['PT_L_FL','PT_L_S1','PT_R_FR','PT_R_S1','S1_A','S1_C','S1_V','Sacrum_Proximal','Sacrum_Distal']
KPS={'PA':PA_KPS,'ML':ML_KPS}
PA_KIND={'lateral sourci':'lateral_sourcil', 'medial sourci':'medial_sourcil',
         'tear drop':'teardrop', 'femoral head':'femoral_head','sacrum':'sacrum'}
ML_KIND={'pt l':'PT_L','pt r':'PT_R','s1':'S1','sacrum':'Sacrum'}
def pt_is_ok(p,w,h):
    return p is not None and len(p)>=3 and p[2]>0 and 0<=p[0]<w and 0<=p[1]<h

def map_ann_to_points(ann,cat,view,w,h):
    name=str(cat['name']).strip().lower()
    kind=(PA_KIND if view=='PA' else ML_KIND).get(name)
    if kind is None: return {}
    keys=cat.get('keypoints', [])
    raw=ann.get('keypoints', [])
    out={}
    for k,key in enumerate(keys):
        if 3*k+2>=len(raw): continue
        suffix=str(key).strip().lower()
        if view=='PA': suffix=suffix.upper(); full=f'{kind}_{suffix}'
        elif kind=='PT_L': full='PT_L_'+('FL' if suffix=='fl' else 'S1' if suffix=='s1' else 'UNKNOWN')
        elif kind=='PT_R': full='PT_R_'+('FR' if suffix=='fr' else 'S1' if suffix=='s1' else 'UNKNOWN')
        elif kind=='S1': full='S1_'+suffix.upper()
        else: full='Sacrum_'+('Proximal' if suffix=='proximal' else 'Distal' if suffix=='distal' else 'UNKNOWN')
        if full in KPS[view]:
            p=list(map(float,raw[3*k:3*k+3]));out[full]=p if pt_is_ok(p,w,h) else [float('nan'),float('nan'),0]
    return out

def frame_id_from_name(s):
    match=re.search(r'mp4-(\d+)',s,re.I)
    return int(match.group(1)) if match else -1

def read_source_zip(path):
    records=[]; manifest=[]; duplicate=[]; missing=[]
    with zipfile.ZipFile(path) as outer:
        for archive in sorted([n for n in outer.namelist() if n.endswith('.zip')]):
            m=re.search(r'--(\d+)\s+(PA|ML)\.coco',archive,re.I)
            if not m: raise ValueError('Cannot decode subject/view from '+archive)
            patient=m.group(1).zfill(3); view=m.group(2).upper()
            with zipfile.ZipFile(io.BytesIO(outer.read(archive))) as inn:
                jnames=[n for n in inn.namelist() if n.endswith('_annotations.coco.json')]
                if len(jnames)!=1: raise ValueError((archive,jnames))
                coco=json.loads(inn.read(jnames[0]));cats={int(c['id']):c for c in coco['categories']}
                anns=defaultdict(list)
                for a in coco['annotations']:anns[int(a['image_id'])].append(a)
                candidates=set(inn.namelist())
                for img in coco['images']:
                    w,h=int(img['width']),int(img['height']); src=img['file_name']; iid=int(img['id'])
                    orig=img.get('extra',{}).get('name',src)
                    fr=frame_id_from_name(orig); dststem=f'{view}_{patient}_{fr:04d}'
                    if fr<0: dststem=f'{view}_{patient}_image{iid:04d}'
                    dst=DATA/f'{dststem}.jpg'
                    if not dst.exists():
                        found=('train/'+src if 'train/'+src in candidates else src)
                        if found not in candidates:raise FileNotFoundError((archive,src))
                        dst.write_bytes(inn.read(found))
                    grouped=defaultdict(list)
                    for a in anns[iid]:
                        cat=cats[int(a['category_id'])]
                        p=map_ann_to_points(a,cat,view,w,h)
                        if p:grouped[str(cat['name']).lower()].append((int(a['id']),p))
                    pts={}
                    for catname, items in grouped.items():
                        if len(items)>1:
                            duplicate.append({'view':view,'patient':patient,'image':dststem,'category':catname,
                                              'n_annotations':len(items),'ids':str([t[0] for t in items])})
                        # Highest number of visible points; fixed deterministic tie-break by smallest ann id
                        chosen=sorted(items,key=lambda t:(-sum(v[2]>0 for v in t[1].values()),t[0]))[0][1]
                        pts.update(chosen)
                    # Image IDs are unique within each video; avoid silent collisions across archives
                    if any(t['stem']==dststem for t in records): raise ValueError('Duplicate source frame: '+dststem)
                    row={'view':view,'patient':patient,'frame':fr,'stem':dststem,'source_name':orig,
                         'file':str(dst),'w':w,'h':h,'pts':pts}
                    records.append(row)
                    missing.append({'view':view,'patient':patient,'image':dststem,'n_visible':sum(pt_is_ok(pts.get(k),w,h) for k in KPS[view]),
                                    'missing':';'.join(k for k in KPS[view] if not pt_is_ok(pts.get(k),w,h))})
            manifest.append({'view':view,'patient':patient,'frames':len(coco['images']),
                             'archive':archive,'person_name':orig.split('_mp4-')[0]})
    return records,pd.DataFrame(manifest),pd.DataFrame(duplicate),pd.DataFrame(missing)

records,manifest,duplicates,presence=read_source_zip(SOURCE_ZIP)
manifest.to_csv(QC/'dataset_manifest.csv',index=False,encoding='utf-8-sig')
duplicates.to_csv(QC/'duplicate_annotations.csv',index=False,encoding='utf-8-sig')
presence.to_csv(QC/'keypoint_presence.csv',index=False,encoding='utf-8-sig')
display(manifest.sort_values(['view','patient']))
print('PA frames:',sum(r['view']=='PA' for r in records),'ML frames:',sum(r['view']=='ML' for r in records))
print('PA patients',sorted(manifest.loc[manifest['view']=='PA','patient'].tolist()))
print('ML patients',sorted(manifest.loc[manifest['view']=='ML','patient'].tolist()))
print('duplicate categories',len(duplicates),'rows with missing keypoints',int(presence.missing.str.len().gt(0).sum()))
assert manifest.loc[manifest['view']=='PA','frames'].sum()==174
assert manifest.loc[manifest['view']=='ML','frames'].sum()==192
assert '005' in manifest.loc[manifest['view']=='PA','patient'].tolist()
assert '005' in manifest.loc[manifest['view']=='ML','patient'].tolist()
print('注意：現ZIPは健常PA 9例、ML 10例。PA 011は含まれません。')


## 3. 患者単位のFold作成・漏洩チェック
GroupKFoldまたはLOVOで患者IDを分割。**同じ患者の別フレームは他の分割に混ぜません**。内側validationはouter train群の1患者に限定します。各foldの `test` はUltralytics標準指標を**学習終了後に独立実行**して測定します。

In [ ]:
def build_folds(view,mode='group5'):
    cases=sorted(set(r['patient'] for r in records if r['view']==view))
    ids=np.array(cases)
    splitter=GroupKFold(n_splits=min(5,len(cases))) if mode=='group5' else LeaveOneGroupOut()
    out=[]
    for fold,(tix,six) in enumerate(splitter.split(ids,groups=ids),start=1):
        outer_train=ids[tix].tolist();test=ids[six].tolist()
        forced=VAL_PATIENT_OVERRIDE.get(view)
        val=forced if forced in outer_train else sorted(outer_train)[-1]
        train=sorted(set(outer_train)-{val})
        assert set(train).isdisjoint(set(test)) and val not in test and val not in train
        assert len(train)>=2
        out.append(dict(view=view,fold=fold,train=train,val=[val],test=sorted(test)))
    return out
FOLDS=[f for v in ('PA','ML') for f in build_folds(v,CV_MODE)]
fold_df=pd.DataFrame([{**{k:f[k] for k in ('view','fold')},**{k:';'.join(f[k]) for k in ('train','val','test')}} for f in FOLDS])
fold_df.to_csv(QC/'patient_fold_manifest.csv',index=False,encoding='utf-8-sig')
display(fold_df)
assert len([f for f in FOLDS if f['view']=='PA' and '005' in f['test']])==1
assert len([f for f in FOLDS if f['view']=='ML' and '005' in f['test']])==1


## 4. YOLO Pose学習データ作成（foldごとに独立）

In [ ]:
def yolo_label(rec, keys, use_points=None):
    pts=rec['pts'] if use_points is None else use_points
    w,h=rec['w'],rec['h']
    visible=[pts.get(k) for k in keys if pt_is_ok(pts.get(k),w,h)]
    if len(visible)<2: return None
    xy=np.array([v[:2] for v in visible],dtype=float)
    # Anatomically bounded object box from all visible kpts
    x0,y0=xy.min(0); x1,y1=xy.max(0); pad=0.08*max(x1-x0,y1-y0,50)
    x0=max(0,x0-pad);x1=min(w,x1+pad);y0=max(0,y0-pad);y1=min(h,y1+pad)
    coords=[0,(x0+x1)/(2*w),(y0+y1)/(2*h),max(2,x1-x0)/w,max(2,y1-y0)/h]
    for k in keys:
        p=pts.get(k)
        if pt_is_ok(p,w,h): coords.extend([np.clip(p[0]/w,0,1),np.clip(p[1]/h,0,1),2])
        else: coords.extend([0,0,0])
    return ' '.join(str(int(x)) if j==0 or (j>=5 and (j-5)%3==2) else f'{x:.7f}' for j,x in enumerate(coords))+'\n'

def create_fold_dataset(spec):
    v=spec['view'];folder=ROOT/'cv_datasets'/f'{CV_MODE}_{v}_fold{spec["fold"]:02d}';keys=KPS[v]
    # rerun-safe: rebuild paths without allowing images from another fold
    for split in ('train','val','test'):
        for sub in ('images','labels'):(folder/sub/split).mkdir(parents=True,exist_ok=True)
        for old in (folder/'images'/split).glob('*.jpg'):old.unlink()
        for old in (folder/'labels'/split).glob('*.txt'):old.unlink()
    omitted=[]
    for split in ('train','val','test'):
        for rec in records:
            if rec['view']!=v or rec['patient'] not in spec[split]:continue
            label=yolo_label(rec,keys)
            if label is None: omitted.append({'fold':spec['fold'],'view':v,'split':split,'patient':rec['patient'],'stem':rec['stem'],'reason':'under_2_labeled_keypoints'});continue
            shutil.copyfile(rec['file'],folder/'images'/split/f'{rec["stem"]}.jpg')
            (folder/'labels'/split/f'{rec["stem"]}.txt').write_text(label)
    with open(folder/'dataset.yaml','w') as f:
        yaml.safe_dump({'path':str(folder),'train':'images/train','val':'images/val','test':'images/test',
                        'names':{0:'pelvis_'+v},'kpt_shape':[len(keys),3],'flip_idx':list(range(len(keys)))},f,sort_keys=False)
    for split in ('train','val','test'):
        assert len(list((folder/'images'/split).glob('*.jpg')))>0,(v,spec['fold'],split)
    return folder/'dataset.yaml',omitted

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def train_pose(data_yaml, dest, epochs, imgsz, batch):
    best=dest/'weights'/'best.pt'
    if best.exists() and not FORCE_RETRAIN:
        print('reuse completed fold:',best);return best
    if not RUN_TRAIN: raise RuntimeError('RUN_TRAIN=Falseで学習済みモデルがありません: '+str(best))
    if best.exists() and FORCE_RETRAIN:shutil.rmtree(dest)
    m=YOLO(MODEL_INITIALIZATION)
    m.train(data=str(data_yaml),project=str(dest.parent),name=dest.name,exist_ok=True,
            epochs=epochs,patience=PATIENCE,imgsz=imgsz,batch=batch,seed=42,
            device=0 if torch.cuda.is_available() else 'cpu',workers=2,cache=False,plots=False,
            degrees=2.0,translate=0.03,scale=0.07,fliplr=0.,flipud=0.,mosaic=0.,mixup=0.,
            pretrained=MODEL_INITIALIZATION.endswith('.pt'),verbose=False)
    del m;clear_gpu()
    if not best.exists():raise RuntimeError(f'best.pt not created: {best}')
    return best

def safe_float(x):
    try:return float(x)
    except (TypeError,ValueError):return np.nan

def get_metrics_row(metrics):
    row={}
    for name in ('pose','box'):
        x=getattr(metrics,name,None)
        for attribute in ('mp','mr','map50','map'):
            row[f'{name}_{"mAP50_95" if attribute=="map" else "mAP50" if attribute=="map50" else "precision" if attribute=="mp" else "recall"}']=safe_float(getattr(x,attribute,None))
    return row

def infer_points(model,rec,imgsz=IMGSZ):
    img=cv2.imread(rec['file'])
    results=model.predict(img,imgsz=imgsz,conf=PRED_CONF,verbose=False,device=0 if torch.cuda.is_available() else 'cpu')
    res=results[0];out={}
    if res.keypoints is None or len(res.keypoints.xy)==0:return out,False,np.nan
    bx=res.boxes.conf.cpu().numpy() if res.boxes is not None else np.ones(len(res.keypoints.xy))
    j=int(np.argmax(bx));xy=res.keypoints.xy[j].cpu().numpy()
    cs=(res.keypoints.conf[j].cpu().numpy() if res.keypoints.conf is not None else np.ones(len(xy)))
    for k,p,cf in zip(KPS[rec['view']],xy,cs):
        if np.isfinite(p).all() and cf>=MIN_KP_CONF and np.linalg.norm(p)>1e-6:
            out[k]=[float(p[0]),float(p[1]),float(cf)]
    return out,True,float(bx[j])


## 5. Group-CVの学習・未知患者評価（再開可能）
このセルで計10モデル（`group5`）または計19モデル（`lovo`）を学習。各foldの **testデータを一切学習に使わず** Pose/Box指標を取得し、test全フレームのkeypoint推論を記録します。完了済foldは再利用。

In [ ]:
fold_metrics=[];pred_rows=[];skipped=[]
for spec in FOLDS:
    v,f=spec['view'],spec['fold']; print('\n===',v,'fold',f,'train=',spec['train'],'val=',spec['val'],'TEST=',spec['test'],'===')
    yaml_path,omit=create_fold_dataset(spec);skipped.extend(omit)
    dest=RUNS/f'{CV_MODE}_{v}_fold{f:02d}'
    metrics_csv=dest/'TEST_external_metrics.csv';pred_csv=dest/'TEST_predictions.csv'
    if metrics_csv.exists() and pred_csv.exists() and not FORCE_RETRAIN:
        fold_metrics.extend(pd.read_csv(metrics_csv).to_dict('records'))
        pred_rows.extend(pd.read_csv(pred_csv).to_dict('records'))
        print('Using prior completed external test outputs');continue
    best=train_pose(yaml_path,dest,EPOCHS,IMGSZ,BATCH)
    m=YOLO(str(best))
    ext=m.val(data=str(yaml_path),split='test',imgsz=IMGSZ,batch=BATCH,
              device=0 if torch.cuda.is_available() else 'cpu',plots=False,
              project=str(dest),name='external_test',exist_ok=True,verbose=False)
    vals={'view':v,'fold':f,'n_test_patients':len(spec['test']),'test_patients':';'.join(spec['test']),
          'n_test_frames':sum(r['view']==v and r['patient'] in spec['test'] for r in records),
          **get_metrics_row(ext)}
    fold_metrics.append(vals);pd.DataFrame([vals]).to_csv(metrics_csv,index=False)
    these=[]
    for rec in tqdm([r for r in records if r['view']==v and r['patient'] in spec['test']],desc=f'{v} fold{f} inference',leave=False):
        pred,det,boxconf=infer_points(m,rec)
        rr={'view':v,'patient':rec['patient'],'frame':rec['frame'],'stem':rec['stem'],
            'fold':f,'detected':int(det),'box_confidence':boxconf}
        for k in KPS[v]:
            pp=pred.get(k)
            rr[f'{k}_x']=pp[0] if pp is not None else np.nan
            rr[f'{k}_y']=pp[1] if pp is not None else np.nan
            rr[f'{k}_conf']=pp[2] if pp is not None else np.nan
        these.append(rr)
    pred_rows.extend(these);pd.DataFrame(these).to_csv(pred_csv,index=False)
    del m,ext;clear_gpu()
pd.DataFrame(skipped).to_csv(QC/'frames_omitted_from_model_training.csv',index=False)
fold_df_out=pd.DataFrame(fold_metrics);oof=pd.DataFrame(pred_rows)
fold_df_out.to_csv(ROOT/'CV_fold_Pose_Box_metrics.csv',index=False,encoding='utf-8-sig')
oof.to_csv(ROOT/'OOF_all_patient_predictions.csv',index=False,encoding='utf-8-sig')
print('Unique OOF frames:',len(oof),'PA',sum(oof['view']=='PA'),'ML',sum(oof['view']=='ML'))
assert len(oof)==len(records),'各frameのOOF予測が1件ずつ必要です。'
assert oof.stem.nunique()==len(records)
display(fold_df_out)


## 6. MAE・RMSE：keypoint誤差と基準線角度誤差（未知患者OOFのみ）
- keypoint `d = √((AIx−GTx)²+(AIy−GTy)²) [px]`、**MAE=mean(d), RMSE=√mean(d²)**。
- 基準線の角度は向きのない直線なので差を `[-90°,90°)` に折り返し、MAE/RMSEはその差の絶対値・二乗で計算。角度名を定義せずに別の線分と混ぜない。
- 欠測や信頼度不足は**エラー0として扱わず、n_validと測定率を併記**。
- 値はフレームの測定誤差であって独立患者数ではない。**症例別CSVとfold別表も作成**。


In [ ]:
PA_LINES={'teardrop_line':('teardrop_R','teardrop_L'),
          'femoral_head_line':('femoral_head_R','femoral_head_L'),
          'medial_sourcil_line':('medial_sourcil_R','medial_sourcil_L'),
          'lateral_sourcil_line':('lateral_sourcil_R','lateral_sourcil_L'),
          'sacrum_line':('sacrum_R','sacrum_L')}
ML_LINES={'PT_L_line':('PT_L_FL','PT_L_S1'), 'PT_R_line':('PT_R_FR','PT_R_S1'),
          'S1_A_C_line':('S1_A','S1_C'),'S1_C_V_line':('S1_C','S1_V'),
          'Sacrum_axis':('Sacrum_Proximal','Sacrum_Distal')}
LINES={'PA':PA_LINES,'ML':ML_LINES}
REC={r['stem']:r for r in records}

def get_pred(pr,k):
    x=pr.get(f'{k}_x',np.nan);y=pr.get(f'{k}_y',np.nan)
    return [float(x),float(y),1] if pd.notna(x) and pd.notna(y) else None

def line_angle(p,q):
    if p is None or q is None:return np.nan
    return math.degrees(math.atan2(q[1]-p[1],q[0]-p[0]))

def axial_angle_diff(a,b):
    return (a-b+90)%180-90 if np.isfinite(a) and np.isfinite(b) else np.nan

kp_errors=[];angle_errors=[]
for _,pr in oof.iterrows():
    r=REC[pr.stem];v=pr['view']
    for k in KPS[v]:
        gt=r['pts'].get(k);ai=get_pred(pr,k)
        if not pt_is_ok(gt,r['w'],r['h']):continue
        d=float(np.linalg.norm(np.array(ai[:2])-np.array(gt[:2]))) if ai is not None else np.nan
        kp_errors.append({'view':v,'patient':pr.patient,'frame':pr.frame,'stem':pr.stem,'fold':pr.fold,
                          'landmark':k,'error_px':d,'abs_dx_px':abs(ai[0]-gt[0]) if ai is not None else np.nan,
                          'abs_dy_px':abs(ai[1]-gt[1]) if ai is not None else np.nan})
    for name,(p,q) in LINES[v].items():
        gp,gq=r['pts'].get(p),r['pts'].get(q)
        if not (pt_is_ok(gp,r['w'],r['h']) and pt_is_ok(gq,r['w'],r['h'])):continue
        pred_p,pred_q=get_pred(pr,p),get_pred(pr,q)
        gt_ang=line_angle(gp,gq);ai_ang=line_angle(pred_p,pred_q)
        signed=axial_angle_diff(ai_ang,gt_ang)
        angle_errors.append({'view':v,'patient':pr.patient,'frame':pr.frame,'stem':pr.stem,'fold':pr.fold,
                             'line':name,'GT_angle_deg':gt_ang,'AI_angle_deg':ai_ang,
                             'signed_error_deg':signed,'abs_error_deg':abs(signed) if np.isfinite(signed) else np.nan})
kp_df=pd.DataFrame(kp_errors);ang_df=pd.DataFrame(angle_errors)
kp_df.to_csv(ROOT/'OOF_keypoint_error_details.csv',index=False,encoding='utf-8-sig')
ang_df.to_csv(ROOT/'OOF_line_angle_error_details.csv',index=False,encoding='utf-8-sig')

def err_summary(d,measure,group_fields):
    def agg(g):
        x=g[measure].dropna().astype(float).values
        return pd.Series({'MAE':float(np.mean(np.abs(x))) if len(x) else np.nan,
                          'RMSE':float(np.sqrt(np.mean(x*x))) if len(x) else np.nan,
                          'n_valid':int(len(x)),'n_GTabs':len(g),
                          'valid_rate':len(x)/len(g) if len(g) else np.nan})
    return d.groupby(group_fields,dropna=False).apply(agg,include_groups=False).reset_index()
for data,col,prefix,target in [(kp_df,'error_px','keypoint','pixel'),(ang_df,'signed_error_deg','angle','degree')]:
    for name,group in [('all',['view']),('item',['view','landmark' if prefix=='keypoint' else 'line']),
                       ('patient',['view','patient']),('fold',['view','fold'])]:
        temp=err_summary(data,col,group);temp['unit']=target
        temp.to_csv(ROOT/f'OOF_{prefix}_{name}_MAE_RMSE.csv',index=False,encoding='utf-8-sig')
    display(err_summary(data,col,['view']))

metric_names=['pose_precision','pose_recall','pose_mAP50','pose_mAP50_95','box_precision','box_recall','box_mAP50','box_mAP50_95']
cv_summary=fold_df_out.groupby('view')[metric_names].agg(['mean','std','count'])
cv_summary.to_csv(ROOT/'CV_pose_box_fold_mean_SD.csv',encoding='utf-8-sig')
display(cv_summary)


## 7. 学会用サマリーFigureと撮影条件確認
`#004 Arimoto ML`などの症例別誤差を確認できます。撮影時の手の重なりはアノテーションや画像を直接確認するまで**原因と断定しません**。

In [ ]:
def save_bar_figure(df,value,name,output,title,unit):
    d=df.copy().sort_values(name)
    fig,ax=plt.subplots(figsize=(10,5))
    ax.bar(np.arange(len(d)),d[value])
    ax.set_xticks(np.arange(len(d)));ax.set_xticklabels(d[name].astype(str),rotation=45,ha='right')
    ax.set_ylabel(unit);ax.set_title(title);ax.grid(axis='y',alpha=.25)
    fig.tight_layout();fig.savefig(FIG/output,dpi=240);plt.close(fig)
for v in ('PA','ML'):
    selected=err_summary(kp_df[kp_df['view']==v],'error_px',['patient'])
    save_bar_figure(selected,'MAE','patient',f'{v}_patient_keypoint_MAE.png',f'{v}: subject-level keypoint MAE (OOF)','pixel')
    selected=err_summary(ang_df[ang_df['view']==v],'signed_error_deg',['line'])
    save_bar_figure(selected,'MAE','line',f'{v}_line_angle_MAE.png',f'{v}: line angle MAE (OOF)','degree')
for v in ('PA','ML'):
    md=fold_df_out[fold_df_out['view']==v]
    fig,ax=plt.subplots(figsize=(8,4))
    xs=np.arange(4); vals=[md[x].mean() for x in ('pose_precision','pose_recall','pose_mAP50','pose_mAP50_95')]
    ax.bar(xs,vals);ax.set_xticks(xs,['Precision','Recall','mAP50','mAP50-95']);ax.set_ylim(0,1.02)
    ax.set_title(f'{v} Pose performance | patient-disjoint {CV_MODE} folds')
    for j,x in enumerate(vals):
        if np.isfinite(x):ax.text(j,x+.015,f'{x:.3f}',ha='center',fontsize=10)
    fig.tight_layout();fig.savefig(FIG/f'{v}_CV_pose_metrics.png',dpi=240);plt.close(fig)
print('figures:',len(list(FIG.glob('*.png'))))


## 8. #005 Matsuda：正面PA／側面ML GTとAI keypoint Figure
**それぞれ#005が外側testだったfoldのbest.pt** で予測した座標を使用。GTとAIを同一フレーム上で色とマーカーで識別。画像選定は原動画フレーム番号の中央値とし、**GTとの誤差が小さい画像を恣意的に選ばない**。PNG 300 dpi・SVGも保存。

In [ ]:
def build_gt_ai_panel(rec,pr,label):
    im=cv2.cvtColor(cv2.imread(rec['file']),cv2.COLOR_BGR2RGB)
    fig,axes=plt.subplots(1,2,figsize=(15,7))
    keys=KPS[rec['view']]
    for ax,title in zip(axes,['Manual annotation (GT)','AI prediction (held-out patient)']):
        ax.imshow(im,cmap='gray');ax.axis('off');ax.set_title(title,fontsize=14)
        for j,k in enumerate(keys):
            p=(rec['pts'].get(k) if ax is axes[0] else get_pred(pr,k))
            if p is None or not pt_is_ok(p,rec['w'],rec['h']):continue
            c=plt.cm.tab10(j%10)
            ax.scatter(p[0],p[1],c=[c],s=42,marker='o' if ax is axes[0] else 'x',linewidths=2)
            ax.text(p[0]+7,p[1]-6,str(j+1),color='white',fontsize=10,
                    bbox={'facecolor':c,'edgecolor':'none','pad':1.3,'alpha':.9})
    fig.suptitle(f'#005 Matsuda {rec["view"]} | frame {rec["frame"]:04d} | out-of-fold model',fontsize=15)
    fig.text(.5,.015,'   '.join(f'{j+1}:{k}' for j,k in enumerate(keys[:5])),ha='center',fontsize=8)
    fig.tight_layout(rect=[0,.045,1,.95]);return fig

for view in ('PA','ML'):
    rs=sorted([r for r in records if r['view']==view and r['patient']=='005'],key=lambda r:r['frame'])
    rec=rs[len(rs)//2];pr=oof[oof.stem==rec['stem']].iloc[0]
    spec=next(f for f in FOLDS if f['view']==view and '005' in f['test'])
    assert '005' not in spec['train']+spec['val']
    f=build_gt_ai_panel(rec,pr,view)
    for ext in ('png','svg'):f.savefig(FIG/f'Matsuda_005_{view}_GT_vs_AI_Keypoints.{ext}',dpi=300,bbox_inches='tight')
    plt.show();plt.close(f)

# 撮影時の手の重なりが疑われる #004 Arimoto ML の注釈・OOF比較をQCとして出力。
view='ML'
rs=sorted([r for r in records if r['view']==view and r['patient']=='004'],key=lambda r:r['frame'])
if rs:
    rec=rs[len(rs)//2];pr=oof[oof.stem==rec['stem']].iloc[0]
    f=build_gt_ai_panel(rec,pr,view)
    f.savefig(FIG/'Arimoto_004_ML_GT_vs_AI_QC.png',dpi=260,bbox_inches='tight')
    plt.close(f)


## 9. #005を除外して専用LCEA用左右局所モデルを学習
全骨盤PA10点モデルは患者外foldのものを使います。さらに8人の非#005 PA画像を、左右対称となる向きに揃えたROIにして **局所3点（骨頭中心、臼蓋外側縁、teardrop）** を学習します。**右ROIは水平反転**し、反転後の局所座標を元画像へ正しく逆変換します。

最終推論ではROI位置、teardrop基準線、LCEAを**すべてAI予測点**から算出し、GTを注入しません。局所予測が失敗した角度は `N/A` とし、成功測定のみのMAEと全測定数を併記します。

In [ ]:
ROI_KEYS=['femoral_head','lateral_sourcil','teardrop']
ROI_SIZE=620
LOCAL_ROOT=ROOT/'Matsuda005_localROI_dataset'

def make_crop(im,cx,cy,size=ROI_SIZE):
    x0=int(round(cx-size/2));y0=int(round(cy-size/2))
    h,w=im.shape[:2];out=np.zeros((size,size,3),dtype=np.uint8)
    sx0=max(x0,0);sy0=max(y0,0);sx1=min(x0+size,w);sy1=min(y0+size,h)
    if sx1>sx0 and sy1>sy0:out[sy0-y0:sy1-y0,sx0-x0:sx1-x0]=im[sy0:sy1,sx0:sx1]
    return out,x0,y0

def make_local_gt(rec,side):
    c=rec['pts'].get('femoral_head_'+side)
    if not pt_is_ok(c,rec['w'],rec['h']):return None
    im=cv2.imread(rec['file']);roi,x0,y0=make_crop(im,c[0],c[1]); pts={}
    for k in ROI_KEYS:
        p=rec['pts'].get(k+'_'+side)
        if not pt_is_ok(p,rec['w'],rec['h']):continue
        x,y=float(p[0]-x0),float(p[1]-y0)
        if side=='R':x=ROI_SIZE-1-x
        if 0<=x<ROI_SIZE and 0<=y<ROI_SIZE:pts[k]=[x,y,2]
    if side=='R':roi=cv2.flip(roi,1)
    return roi,pts

def prepare_local_data():
    folder=LOCAL_ROOT
    patients=sorted(set(r['patient'] for r in records if r['view']=='PA' and r['patient']!='005'))
    val_id='010' if '010' in patients else patients[-1]
    counts=Counter()
    for rec in records:
        if rec['view']!='PA' or rec['patient']=='005':continue
        split='val' if rec['patient']==val_id else 'train'
        for sub in ('images','labels'):(folder/sub/split).mkdir(parents=True,exist_ok=True)
        for side in ('R','L'):
            val=make_local_gt(rec,side)
            if val is None:continue
            img,pts=val
            localrec={'w':ROI_SIZE,'h':ROI_SIZE,'pts':pts}
            lab=yolo_label(localrec,ROI_KEYS)
            if lab is None:continue
            stem=f'{rec["stem"]}_{side}'
            cv2.imwrite(str(folder/'images'/split/f'{stem}.jpg'),img)
            (folder/'labels'/split/f'{stem}.txt').write_text(lab)
            counts[split]+=1
    assert counts['train']>10 and counts['val']>2,counts
    data_yaml=folder/'dataset.yaml'
    yaml.safe_dump({'path':str(folder),'train':'images/train','val':'images/val',
                    'names':{0:'hip_ROI'},'kpt_shape':[3,3],'flip_idx':[0,1,2]},
                   open(data_yaml,'w'),sort_keys=False)
    return data_yaml,counts

local_pt=None
if RUN_LOCAL_005:
    local_yaml,local_counts=prepare_local_data();print('local ROI samples:',local_counts)
    local_pt=train_pose(local_yaml,RUNS/'Matsuda005_localROI_3kp',LOCAL_EPOCHS,LOCAL_IMGSZ,LOCAL_BATCH)


## 10. #005 Matsuda PA：LCEA AI計測値入り動画・CSV
LCEAは、**AIの左右teardropを結ぶ水平基準から垂直上向き**と、AI予測骨頭中心→臼蓋外側縁のなす角として定義（右左の外側方向を統一）。角度の正負、ランドマーク位置、overlayを医学的に最終目視確認してください。

標準入力は**COCOに入った#005 Matsuda PAの21静止フレームをフレーム番号順で再構成したデモMP4**です。元動画そのものではなく、指定 `DISPLAY_FPS` は表示用。元の実DDR mp4を別途持っている場合は `ORIGINAL_005_PA_MP4` に指定すればオリジナルのfpsで処理します。


In [ ]:
def bbox_best_result(model,im,imgsz,keys):
    res=model.predict(im,imgsz=imgsz,conf=PRED_CONF,verbose=False,
                      device=0 if torch.cuda.is_available() else 'cpu')[0]
    if res.keypoints is None or len(res.keypoints.xy)==0:return {}
    bs=res.boxes.conf.cpu().numpy() if res.boxes is not None else np.ones(len(res.keypoints.xy))
    j=int(np.argmax(bs));xy=res.keypoints.xy[j].cpu().numpy()
    cf=res.keypoints.conf[j].cpu().numpy() if res.keypoints.conf is not None else np.ones(len(xy))
    return {k:[float(x),float(y),float(c)] for k,(x,y),c in zip(keys,xy,cf) if c>=MIN_KP_CONF and (abs(x)+abs(y)>1.e-5)}

def local_on_global(img,gpts,local_model,side):
    center=gpts.get('femoral_head_'+side)
    if center is None:return {},'global_femoral_head_missing'
    roi,x0,y0=make_crop(img,center[0],center[1]);roi=cv2.flip(roi,1) if side=='R' else roi
    local=bbox_best_result(local_model,roi,LOCAL_IMGSZ,ROI_KEYS)
    out={}
    for key,p in local.items():
        x=(ROI_SIZE-1-p[0]) if side=='R' else p[0]
        x=x+x0;y=p[1]+y0
        if 0<=x<img.shape[1] and 0<=y<img.shape[0]:out[f'{key}_{side}']=[x,y,p[2]]
    return out,''

def lcea_from_ai(pts,side):
    R,L=pts.get('teardrop_R'),pts.get('teardrop_L')
    head=pts.get('femoral_head_'+side);roof=pts.get('lateral_sourcil_'+side)
    if any(p is None for p in (R,L,head,roof)):return np.nan,'missing_points'
    axis=np.array(L[:2])-np.array(R[:2]);norm=np.linalg.norm(axis)
    v=np.array(roof[:2])-np.array(head[:2])
    if norm<100 or np.linalg.norm(v)<10:return np.nan,'degenerate_geometry'
    u=axis/norm;up=np.array([u[1],-u[0]])
    # geometric sanity: roof should lie superior and somewhat lateral relative to head
    vertical=np.dot(v,up); lateral=np.dot(v,u)*(1 if side=='L' else -1)
    if vertical<=0:return np.nan,'roof_not_superior'
    angle=math.degrees(math.atan2(lateral,vertical))
    if not -30<=angle<=85:return np.nan,'angle_out_of_QC_range'
    return angle,'ok'

def draw_frame(img,gpts,roi_pts,angles,idx):
    overlay=img.copy();h,w=overlay.shape[:2]
    # global pelvic horizontal baseline
    a,b=gpts.get('teardrop_R'),gpts.get('teardrop_L')
    if a is not None and b is not None:
        cv2.line(overlay,tuple(map(int,a[:2])),tuple(map(int,b[:2])),(255,230,0),2,cv2.LINE_AA)
    for side,col in (('R',(60,215,255)),('L',(60,255,110))):
        hp=roi_pts.get('femoral_head_'+side);roof=roi_pts.get('lateral_sourcil_'+side)
        if hp is not None:cv2.circle(overlay,tuple(map(int,hp[:2])),7,col,-1,cv2.LINE_AA)
        if roof is not None:cv2.circle(overlay,tuple(map(int,roof[:2])),7,(0,140,255),-1,cv2.LINE_AA)
        if hp is not None and roof is not None:cv2.line(overlay,tuple(map(int,hp[:2])),tuple(map(int,roof[:2])),col,3,cv2.LINE_AA)
    cv2.rectangle(overlay,(6,6),(min(w-6,640),132),(0,0,0),-1)
    cv2.putText(overlay,'Matsuda 005 PA | AI LCEA (held-out)',(20,35),cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),2,cv2.LINE_AA)
    for side,y in (('R',70),('L',110)):
        val,why=angles[side]
        txt=f'{side}: {val:.1f} deg' if np.isfinite(val) else f'{side}: N/A ({why})'
        cv2.putText(overlay,txt,(20,y),cv2.FONT_HERSHEY_SIMPLEX,.85,(200,245,255) if side=='R' else (180,255,180),2,cv2.LINE_AA)
    return overlay

if RUN_LOCAL_005:
    pa_spec=next(s for s in FOLDS if s['view']=='PA' and '005' in s['test'])
    assert '005' not in pa_spec['train'] and '005' not in pa_spec['val']
    global005=YOLO(str(RUNS/f'{CV_MODE}_PA_fold{pa_spec["fold"]:02d}'/'weights'/'best.pt'))
    local005=YOLO(str(local_pt))
    def predict_one_lcea(img):
        g=bbox_best_result(global005,img,IMGSZ,PA_KPS)
        refined={};reasons={}
        for side in ('R','L'):
            new,why=local_on_global(img,g,local005,side)
            refined.update(new);reasons[side]=why
        for k in ('teardrop_R','teardrop_L'):
            if k in g:refined[k]=g[k]
        return g,refined,{side:lcea_from_ai(refined,side) for side in ('R','L')}

    default_frames=sorted([r for r in records if r['view']=='PA' and r['patient']=='005'],key=lambda r:r['frame'])
    has_original=(ORIGINAL_005_PA_MP4 is not None and Path(ORIGINAL_005_PA_MP4).exists())
    if has_original:
        cap=cv2.VideoCapture(str(ORIGINAL_005_PA_MP4))
        fps=float(cap.get(cv2.CAP_PROP_FPS)); fps=fps if 0<fps<500 else DISPLAY_FPS
        nframes=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        source='original_video'
        def iter_frames():
            idx=0
            while True:
                ok,im=cap.read()
                if not ok:break
                yield idx,im,None
                idx+=1
    else:
        fps=DISPLAY_FPS;source='annotated_image_sequence'
        def iter_frames():
            for idx,r in enumerate(default_frames): yield idx,cv2.imread(r['file']),r
    writer=None;angle_rows=[]
    mp4_out=VIDEOS/'Matsuda005_PA_AI_LCEA.mp4'
    for frame_i,img,r in tqdm(iter_frames(),total=(nframes if has_original and nframes>0 else len(default_frames))):
        g,refined,angles=predict_one_lcea(img)
        show=draw_frame(img,g,refined,angles,frame_i)
        if writer is None:
            h,w=show.shape[:2]
            writer=cv2.VideoWriter(str(mp4_out),cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h))
            if not writer.isOpened():raise RuntimeError('Could not open mp4 writer')
        writer.write(show)
        row={'frame_number':frame_i,'original_annotation_frame':r['frame'] if r is not None else np.nan,'source':source,
             'GT_inference_leakage':False,'global_model_excluded_005':True,'local_model_excluded_005':True}
        for side in ('R','L'):
            deg,reason=angles[side]
            row[f'LCEA_AI_{side}_deg']=deg;row[f'LCEA_QC_{side}']=reason
            # 参考：手動基準値は別列の事後評価のみ、AIの推論には使わない
            if r is not None:
                manual={k:p for k,p in r['pts'].items()}
                val,_=lcea_from_ai(manual,side)
                row[f'LCEA_GT_{side}_deg']=val
                row[f'LCEA_AI_minus_GT_{side}_deg']=deg-val if np.isfinite(deg) and np.isfinite(val) else np.nan
        angle_rows.append(row)
    if writer is not None:writer.release()
    if has_original:cap.release()
    # PowerPoint等での再生互換性のためH.264 (yuv420p)へ変換。ffmpegがない場合はmp4v版を維持。
    h264=VIDEOS/'Matsuda005_PA_AI_LCEA_H264.mp4'
    if shutil.which('ffmpeg') and mp4_out.exists():
        cmd=['ffmpeg','-y','-loglevel','error','-i',str(mp4_out),'-c:v','libx264',
             '-pix_fmt','yuv420p','-crf','20',str(h264)]
        ff=subprocess.run(cmd,capture_output=True,text=True)
        if ff.returncode!=0:
            warnings.warn('H.264 conversion failed; MP4V version available: '+ff.stderr[-250:])
        else: print('PowerPoint-compatible H.264:',h264)
    angles005=pd.DataFrame(angle_rows)
    angles005.to_csv(VIDEOS/'Matsuda005_PA_LCEA_AI_and_GT.csv',index=False,encoding='utf-8-sig')
    print('Saved',mp4_out,'frames',len(angles005),'file size',mp4_out.stat().st_size)
    print('AI valid measurement fractions:')
    for side in ('R','L'):
        n=angles005[f'LCEA_AI_{side}_deg'].notna().sum()
        print(side,n,'/',len(angles005),' valid')
        if not has_original:
            err=angles005[f'LCEA_AI_minus_GT_{side}_deg'].dropna().values
            print('MAE',np.mean(np.abs(err)) if len(err) else 'NA','RMSE',np.sqrt(np.mean(err*err)) if len(err) else 'NA')
    if not has_original:print('これはannotation画像列から作った表示用動画で、元DDR動画の実FPSを再現したものではありません。')
    display(angles005.head())
    del global005,local005;clear_gpu()


## 11. 追加：LCEA Figureと学会用最終表
**#005のAI角度とGT角度の時系列**を保存。PA/ML Pose 4指標と位置・角度のMAE/RMSEが別列で出力される。

In [ ]:
if RUN_LOCAL_005 and 'angles005' in globals() and not angles005.empty and 'LCEA_GT_R_deg' in angles005.columns:
    fig,ax=plt.subplots(figsize=(10,5))
    for side,color in (('R','tab:orange'),('L','tab:blue')):
        x=angles005.original_annotation_frame
        ax.plot(x,angles005[f'LCEA_GT_{side}_deg'],color=color,ls='--',lw=1.5,label=f'{side} manual GT')
        ax.plot(x,angles005[f'LCEA_AI_{side}_deg'],color=color,marker='o',lw=1.6,label=f'{side} AI')
    ax.set(xlabel='Annotated source frame index (not seconds)',ylabel='LCEA (degree)',title='#005 Matsuda PA | OOF AI LCEA vs manual reference')
    ax.grid(alpha=.2);ax.legend(ncol=2);fig.tight_layout()
    for ext in ('png','svg'):fig.savefig(FIG/f'Matsuda005_PA_LCEA_AI_vs_GT.{ext}',dpi=300)
    plt.show();plt.close(fig)

# 1 row per view: mean±SD fold metrics and pooled point/angle errors (definitions differing)
summary=[]
for v in ('PA','ML'):
    vm=fold_df_out[fold_df_out['view']==v]
    k=kp_df[kp_df['view']==v].error_px.dropna().values
    a=ang_df[ang_df['view']==v].signed_error_deg.dropna().values
    row={'view':v,'patients':int(manifest[manifest['view']==v].patient.nunique()),
         'frames':int(manifest[manifest['view']==v].frames.sum()),
         'n_cv_folds':len(vm),
         'kpt_MAE_px':float(np.mean(k)) if len(k) else np.nan,
         'kpt_RMSE_px':float(np.sqrt(np.mean(k*k))) if len(k) else np.nan,
         'kpt_valid':len(k),'kpt_GT':int((kp_df['view']==v).sum()),
         'angle_MAE_deg':float(np.mean(np.abs(a))) if len(a) else np.nan,
         'angle_RMSE_deg':float(np.sqrt(np.mean(a*a))) if len(a) else np.nan,
         'angle_valid':len(a),'angle_GT':int((ang_df['view']==v).sum())}
    for m in metric_names:
        row[m+'_fold_mean']=float(vm[m].mean());row[m+'_fold_SD']=float(vm[m].std())
    summary.append(row)
pd.DataFrame(summary).to_csv(ROOT/'CONFERENCE_PA_ML_final_summary.csv',index=False,encoding='utf-8-sig')
display(pd.DataFrame(summary).T)


## 12. 成果物ZIPにまとめてダウンロード
含める：症例リスト、split監査、全fold評価、全keypoint/角度MAE・RMSE、患者別結果、#005 keypoint比較Figure（PA+ML）、#005 PA AI-LCEA動画、LCEAフレームCSV、各foldの `best.pt`（再学習・再描画用）。大量の元画像はZIPに含めません。

**学会前の最終QA:** ①PAのNは9である、②#004 Arimoto側面は画像QCを実施、③#005の点はtest OOF予測、④LCEAの手動点を推論時に使用していない、⑤LCEA角度にN/Aを含めた成功率を併記、⑥画像列デモ動画を「元DDR動画」と呼ばない。

In [ ]:
final_files=[]
for x in ROOT.rglob('*'):
    if not x.is_file():continue
    path=x.relative_to(ROOT)
    if 'cv_datasets' in path.parts or 'source' in path.parts or 'Matsuda005_localROI_dataset' in path.parts:continue
    if '/weights/' in str(path).replace('\\','/') and x.name!='best.pt':continue
    if x.suffix.lower() in ('.jpg','.jpeg') and path.parts[0] not in ('figures',):continue
    if any(x.name==n for n in ('last.pt',)):continue
    if x.stat().st_size>65_000_000:continue
    final_files.append(x)
with zipfile.ZipFile(OUTPUT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=2) as z:
    for p in final_files:z.write(p,arcname=str(p.relative_to(ROOT)))
print('Output:',OUTPUT_ZIP,'size MB:',round(OUTPUT_ZIP.stat().st_size/1e6,1),'files:',len(final_files))
try:
    from google.colab import files
    files.download(str(OUTPUT_ZIP))
except ImportError:
    pass


### 学会発表用・解析説明テンプレート
- 「健常DDRの**PA 9患者・174フレーム、ML 10患者・192フレーム**を解析した」（現ZIPに存在するものだけ）。
- 「PA10点、ML9点のYOLO Poseモデルを新規に学習した。フレーム単位の漏洩を防ぐため、患者・動画単位で **5-fold GroupKFold**（またはLOVO）を実施した。」
- 「各foldで**未学習test患者**の Pose precision/recall/mAP50/mAP50-95、keypointユークリッド位置誤差と所定の2点線分の角度誤差について、MAE/RMSEを算出した。」
- 「#005 Matsuda PAでは未学習のglobal10点モデルと右左統一の局所3点モデルからLCEAを連続計測した。」
- 「アノテーションされた静止フレームから再構成した動画であり、元撮影の時間分解能を示すものではない（元mp4使用時を除く）。」

**既報5例のmAPと新規9/10例のmAPは、症例構成・分割方法の違いを明記して並列提示し、厳密な『モデル改善』の統計的証明とは扱わない。**
